In [2]:
import pandas as pd
import sqlite3
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))
from config import DB_FILE

conn = sqlite3.connect(DB_FILE)


First I load the data I need for all Visualizations.

In [ ]:
#For the line chart and waterfall / annual P&L summary

annual = pd.read_sql("""
    SELECT
        Calendar.Year,
        COA.Class,
        COA.SubClass,
        SUM(gl.Amount) AS Amount
    FROM GL
    JOIN COA      ON GL.Account_key = COA.Account_key
    JOIN Calendar ON GL.Date        = Calendar.Date
    WHERE COA.Report = 'Profit and Loss'
    GROUP BY Calendar.Year, COA.Class, COA.SubClass
""", conn)

#For the bar chart / P&L by region
regional = pd.read_sql("""
    SELECT
        calenCalendardar.Year,
        territory.Region,
        COA.SubClass,
        SUM(gl.Amount) AS Amount
    FROM GL
    JOIN COA       ON GL.Account_key   = COA.Account_key
    JOIN Calendar  ON GL.Date          = Calendar.Date
    JOIN territory ON GL.Territory_key = territory.Territory_key
    WHERE COA.Report = 'Profit and Loss'
    GROUP BY Calendar.Year, territory.Region, COA.SubClass
""", conn)

#For the heatmap / monthly expenses
heatmap_data = pd.read_sql("""
    SELECT
        Calendar.Year,
        Calendar.Month,
        COA.SubClass,
        SUM(GL.Amount) AS Amount
    FROM GL
    JOIN COA      ON GL.Account_key = COA.Account_key
    JOIN Calendar ON GL.Date        = Calendar.Date
    WHERE COA.SubClass IN (
        'Operating Expenses',
        'Depreciation & Amortization',
        'Interest Expense',
        'Taxation'
    )
    GROUP BY Calendar.Year, Calendar.Month, COA.SubClass
""", conn)

print("Data loaded")
print(f"Annual rows:   {len(annual)}")
print(f"Regional rows: {len(regional)}")
print(f"Heatmap rows:  {len(heatmap_data)}")